In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────
!pip install -q streamlit pyngrok transformers torch
!pip install -q tensorflow pillow joblib scikit-learn spacy
!python -m spacy download en_core_web_sm
print('✅ All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 50.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ All packages installed


In [ ]:
# ── Cell 2: Mount Drive + verify all files exist ─────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

BASE = '/content/drive/MyDrive/fdp'

required = {
    'ml_module.py'              : f'{BASE}/ml_module.py',
    'dl_module.py'              : f'{BASE}/dl_module.py',
    'nlp_module.py'             : f'{BASE}/nlp_module.py',
    'llm_module.py'             : f'{BASE}/llm_module.py',
    'ml_model.pkl'              : f'{BASE}/ml_model/ml_model.pkl',
    'ml_scaler.pkl'             : f'{BASE}/ml_model/ml_scaler.pkl',
    'dl_model.keras'            : f'{BASE}/dl_model/dl_model.keras',
    'nlp model.safetensors'     : f'{BASE}/nlp_model/model.safetensors',
    'llm model.safetensors'     : f'{BASE}/llm_model/model.safetensors',
}

print('📁 Checking all required files:\n')
all_ok = True
for name, path in required.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1024/1024 if exists else 0
    status = '✅' if exists else '❌'
    print(f'   {status}  {name:30s}  {size:.1f} MB')
    if not exists:
        all_ok = False

print(f'\n{"✅ All files found — ready for Day 5!" if all_ok else "❌ Some files missing — check Drive"}')

Mounted at /content/drive
📁 Checking all required files:

   ✅  ml_module.py                    0.0 MB
   ✅  dl_module.py                    0.0 MB
   ✅  nlp_module.py                   0.0 MB
   ✅  llm_module.py                   0.0 MB
   ✅  ml_model.pkl                    2.4 MB
   ✅  ml_scaler.pkl                   0.0 MB
   ✅  dl_model.keras                  22.7 MB
   ✅  nlp model.safetensors           255.4 MB
   ✅  llm model.safetensors           2987.5 MB

✅ All files found — ready for Day 5!


In [ ]:
# ── Cell 3: Copy module files to Colab working directory ──────────
import shutil, os, sys

WORK_DIR = '/content/fdp_app'
os.makedirs(WORK_DIR, exist_ok=True)

BASE = '/content/drive/MyDrive/fdp'

# Copy all module files
for module in ['ml_module.py', 'dl_module.py', 'nlp_module.py', 'llm_module.py']:
    shutil.copy(f'{BASE}/{module}', f'{WORK_DIR}/{module}')
    print(f'✅ Copied {module}')

# Add working dir to Python path
sys.path.insert(0, WORK_DIR)
print(f'\n✅ Working directory ready: {WORK_DIR}')

✅ Copied ml_module.py
✅ Copied dl_module.py
✅ Copied nlp_module.py
✅ Copied llm_module.py

✅ Working directory ready: /content/fdp_app


In [ ]:
# ── Cell 4: Integration test — all 4 modules ─────────────────────
import sys
sys.path.insert(0, '/content/fdp_app')

print('Loading all 4 modules...\n')

# ── ML ────────────────────────────────────────────────────────────
import joblib
import pandas as pd

ml_model  = joblib.load('/content/drive/MyDrive/fdp/ml_model/ml_model.pkl')
ml_scaler = joblib.load('/content/drive/MyDrive/fdp/ml_model/ml_scaler.pkl')

def predict_risk(age, bp, chol, sugar, hr, sex,
                 cp=0, ecg=0, exang=0, oldpeak=1.0, slope=1, ca=0, thal=2):
    try:
        df = pd.DataFrame([{
            'Age': age,
            'Sex': sex,
            'ChestPainType': cp,
            'RestingBP': bp,
            'Cholesterol': chol,
            'FastingBS': sugar,
            'RestingECG': ecg,
            'MaxHR': hr,
            'ExerciseAngina': exang,
            'Oldpeak': oldpeak,
            'ST_Slope': slope
        }])

        # Ensure correct feature order
        if hasattr(ml_scaler, "feature_names_in_"):
            df = df[ml_scaler.feature_names_in_]

        df = ml_scaler.transform(df)
        score = ml_model.predict_proba(df)[0][1]

        label = 'HIGH' if score > 0.6 else 'MODERATE' if score > 0.35 else 'LOW'
        return {'label': label, 'score': round(float(score), 2)}

    except Exception as e:
        print("❌ ML Error:", e)
        return {'label': 'ERROR', 'score': 0.0}


# ✅ RUN ML (FIXED)
print("\nRunning ML module...")
try:
    risk = predict_risk(55, 160, 280, 140, 95, 1)
    print(f'✅ ML  : {risk}')
except Exception as e:
    print("❌ ML Failed:", e)
    risk = {'label': 'UNKNOWN', 'score': 0.0}


# ── DL ────────────────────────────────────────────────────────────
import tensorflow as tf
from PIL import Image
import io as _io
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

dl_model  = tf.keras.models.load_model(
    '/content/drive/MyDrive/fdp/dl_model/dl_model.keras'
)
CLASS_MAP = {0: 'NORMAL', 1: 'PNEUMONIA'}

def classify_xray(img_bytes):
    try:
        img  = Image.open(_io.BytesIO(img_bytes)).convert('RGB').resize((224, 224))
        arr  = np.array(img, dtype=np.float32)
        arr  = preprocess_input(arr)
        arr  = np.expand_dims(arr, axis=0)
        pred = dl_model.predict(arr, verbose=0)
        idx  = int(np.argmax(pred))
        return {'label': CLASS_MAP[idx], 'confidence': round(float(np.max(pred)), 2)}
    except Exception as e:
        print("❌ DL Error:", e)
        return {'label': 'Error', 'confidence': 0.0}


# Test DL
buf = _io.BytesIO()
Image.new('RGB', (224, 224), color='white').save(buf, 'JPEG')
xray = classify_xray(buf.getvalue())
print(f'✅ DL  : {xray}')


# ── NLP ───────────────────────────────────────────────────────────
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLP_PATH  = '/content/drive/MyDrive/fdp/nlp_model'
NLP_LABELS = {0: 'CRITICAL', 1: 'MODERATE', 2: 'MILD'}

SYMPTOM_WORDS = {
    'chest pain', 'breathlessness', 'fever', 'vomiting', 'headache',
    'dizziness', 'nausea', 'weakness', 'cough', 'rash', 'swelling',
    'paralysis', 'seizure', 'unconscious', 'sweating', 'bleeding',
    'burning', 'fatigue', 'palpitations', 'wheezing', 'confusion'
}

nlp_device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nlp_tokenizer = AutoTokenizer.from_pretrained(NLP_PATH)
nlp_model     = AutoModelForSequenceClassification.from_pretrained(NLP_PATH).to(nlp_device)
nlp_model.eval()

def extract_symptoms(text):
    text = text.lower()
    return list({w for w in SYMPTOM_WORDS if re.search(rf"\b{w}\b", text)})

def analyze_complaint(text):
    try:
        enc = nlp_tokenizer([text], padding=True, truncation=True,
                            max_length=128, return_tensors='pt')
        enc = {k: v.to(nlp_device) for k, v in enc.items()}
        with torch.no_grad():
            logits = nlp_model(**enc).logits
        pred_id = int(torch.argmax(logits))
        return {'symptoms': extract_symptoms(text), 'urgency': NLP_LABELS[pred_id]}
    except Exception as e:
        print("❌ NLP Error:", e)
        return {'symptoms': [], 'urgency': 'UNKNOWN'}

# Test NLP
nlp = analyze_complaint('Severe chest pain and breathlessness since 2 hours')
print(f'✅ NLP : {nlp}')


# ── LLM ───────────────────────────────────────────────────────────
from transformers import AutoTokenizer as LLMTok, AutoModelForSeq2SeqLM

LLM_PATH  = '/content/drive/MyDrive/fdp/llm_model'
llm_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

llm_tok   = LLMTok.from_pretrained(LLM_PATH)
llm_mdl   = AutoModelForSeq2SeqLM.from_pretrained(LLM_PATH).to(llm_device)
llm_mdl.eval()

def generate_report(name, age, risk, xray, nlp_result):
    syms = ', '.join(nlp_result['symptoms']) if nlp_result['symptoms'] else 'none reported'
    prompt = (
        f'Patient {name}, age {age}. Risk: {risk["label"]} ({risk["score"]:.0%}). '
        f'X-ray: {xray["label"]} ({xray["confidence"]:.0%}). '
        f'Symptoms: {syms}. Urgency: {nlp_result["urgency"]}. '
        f'Write a 5-sentence clinical summary and recommend action.'
    )
    try:
        inp = llm_tok(prompt, return_tensors='pt',
                      truncation=True, max_length=600).to(llm_device)
        with torch.no_grad():
            out = llm_mdl.generate(**inp, max_new_tokens=300, min_new_tokens=80,
                                   num_beams=5, length_penalty=2.0,
                                   no_repeat_ngram_size=3, repetition_penalty=1.3)
        report = llm_tok.decode(out[0], skip_special_tokens=True).strip()
        if len(report.split()) < 20:
            raise ValueError('Too short')
        return {'report': report}
    except Exception:
        action = {'CRITICAL': 'Immediate emergency care required.',
                  'MODERATE': 'Admit for monitoring and treatment.',
                  'MILD':     'Discharge with outpatient follow-up.'
                  }.get(nlp_result['urgency'], 'Clinical evaluation recommended.')
        return {'report': f'Patient {name}, {age} years, presents with '
                          f'{risk["label"]} risk and {xray["label"]} on imaging. '
                          f'Symptoms: {syms}. Urgency: {nlp_result["urgency"]}. {action}'}

# Test LLM
rep = generate_report('Ramesh Kumar', 55, risk, xray, nlp)
print(f'✅ LLM : {rep["report"][:80]}...')

print('\n🎉 All 4 modules working — ready to build Streamlit app!')

Loading all 4 modules...


Running ML module...
✅ ML  : {'label': 'HIGH', 'score': 0.86}


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 40 variables whereas the saved optimizer has 78 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✅ DL  : {'label': 'PNEUMONIA', 'confidence': 0.69}


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ NLP : {'symptoms': ['chest pain', 'breathlessness'], 'urgency': 'MODERATE'}


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ LLM : Ramesh Kumar, age 55 has chest pain and breathlessness. He should see a doctor r...

🎉 All 4 modules working — ready to build Streamlit app!


In [ ]:
# ── Cell 5: Write agent.py ────────────────────────────────────────
agent_code = '''# Day 5 — Agent | Piyush Pandey
import joblib, torch, numpy as np, io
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForSeq2SeqLM)

BASE = "/content/drive/MyDrive/fdp"

# ── Load all models once at import time ──────────────────────────
_ml_model  = joblib.load(f"{BASE}/ml_model/ml_model.pkl")
_ml_scaler = joblib.load(f"{BASE}/ml_model/ml_scaler.pkl")
_dl_model  = tf.keras.models.load_model(f"{BASE}/dl_model/dl_model.keras")
_dev       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_nlp_tok   = AutoTokenizer.from_pretrained(f"{BASE}/nlp_model")
_nlp_mdl   = AutoModelForSequenceClassification.from_pretrained(
                f"{BASE}/nlp_model").to(_dev)
_nlp_mdl.eval()

_llm_tok   = AutoTokenizer.from_pretrained(f"{BASE}/llm_model")
_llm_mdl   = AutoModelForSeq2SeqLM.from_pretrained(
                f"{BASE}/llm_model").to(_dev)
_llm_mdl.eval()

DL_CLASSES  = {0: "NORMAL", 1: "PNEUMONIA"}
NLP_LABELS  = {0: "CRITICAL", 1: "MODERATE", 2: "MILD"}
SYMPTOM_WORDS = {
    "chest pain","breathlessness","fever","vomiting","headache",
    "dizziness","nausea","weakness","cough","rash","swelling",
    "paralysis","seizure","unconscious","sweating","bleeding",
    "burning","fatigue","palpitations","wheezing","confusion"
}

class TriageAgent:
    def run(self, name, age, bp, chol, sugar, hr, sex,
            complaint, img_bytes=None):
        result = {"name": name, "age": age}

        # Module 1 — ML Risk
        try:
            feat  = [[age, sex, 0, bp, chol, sugar, 0, hr, 0, 1.0, 1, 0, 2]]
            feat  = _ml_scaler.transform(feat)
            score = _ml_model.predict_proba(feat)[0][1]
            label = "HIGH" if score > 0.6 else "MODERATE" if score > 0.35 else "LOW"
            result["risk"] = {"label": label, "score": round(float(score), 2)}
        except Exception as e:
            result["risk"] = {"label": "UNKNOWN", "score": 0.0}

        # Module 2 — DL X-ray (optional)
        if img_bytes:
            try:
                img  = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((224,224))
                arr  = preprocess_input(np.array(img, dtype=np.float32))
                arr  = np.expand_dims(arr, 0)
                pred = _dl_model.predict(arr, verbose=0)
                idx  = int(np.argmax(pred))
                result["xray"] = {"label": DL_CLASSES[idx],
                                  "confidence": round(float(np.max(pred)), 2)}
            except Exception:
                result["xray"] = {"label": "Error", "confidence": 0.0}
        else:
            result["xray"] = {"label": "Not provided", "confidence": 0.0}

        # Module 3 — NLP Complaint
        try:
            enc  = _nlp_tok([complaint], padding=True, truncation=True,
                            max_length=128, return_tensors="pt")
            enc  = {k: v.to(_dev) for k, v in enc.items()}
            with torch.no_grad():
                logits = _nlp_mdl(**enc).logits
            pid  = int(torch.argmax(logits))
            syms = [w for w in SYMPTOM_WORDS if w in complaint.lower()]
            result["nlp"] = {"symptoms": list(set(syms)),
                             "urgency": NLP_LABELS[pid]}
        except Exception:
            result["nlp"] = {"symptoms": [], "urgency": "UNKNOWN"}

        # Module 4 — LLM Report
        try:
            syms_str = ", ".join(result["nlp"]["symptoms"]) or "none reported"
            prompt   = (
                f"Patient {name}, age {age}. "
                f"Risk: {result[\\'risk\\'][\\'label\\']} ({result[\\'risk\\'][\\'score\\']:.0%}). "
                f"X-ray: {result[\\'xray\\'][\\'label\\']} ({result[\\'xray\\'][\\'confidence\\']:.0%}). "
                f"Symptoms: {syms_str}. Urgency: {result[\\'nlp\\'][\\'urgency\\']}. "
                f"Write a 5-sentence clinical summary and recommend action."
            )
            inp = _llm_tok(prompt, return_tensors="pt",
                           truncation=True, max_length=600).to(_dev)
            with torch.no_grad():
                out = _llm_mdl.generate(**inp, max_new_tokens=300,
                                        min_new_tokens=80, num_beams=5,
                                        length_penalty=2.0, no_repeat_ngram_size=3,
                                        repetition_penalty=1.3)
            report = _llm_tok.decode(out[0], skip_special_tokens=True).strip()
            if len(report.split()) < 20:
                raise ValueError("Too short")
            result["report"] = report
        except Exception:
            action = {"CRITICAL": "Immediate emergency care required.",
                      "MODERATE": "Admit for monitoring and treatment.",
                      "MILD":     "Discharge with outpatient follow-up."
                      }.get(result["nlp"]["urgency"], "Clinical evaluation recommended.")
            result["report"] = (
                f"Patient {name}, {age} years, presents with "
                f"{result[\\'risk\\'][\\'label\\'].lower()} cardiovascular risk "
                f"and {result[\\'xray\\'][\\'label\\'].lower()} on chest imaging. "
                f"Urgency: {result[\\'nlp\\'][\\'urgency\\']}. {action}"
            )
        return result
'''

with open(f'{WORK_DIR}/agent.py', 'w') as f:
    f.write(agent_code)
print('✅ agent.py written')

✅ agent.py written


In [ ]:
# ── Cell 6: Write app.py ──────────────────────────────────────────
app_code = '''# Day 5 — Streamlit App | Piyush Pandey
import streamlit as st
import io
from agent import TriageAgent

st.set_page_config(
    page_title="AI Medical Triage Assistant",
    page_icon="🏥",
    layout="wide"
)

# ── Header ────────────────────────────────────────────────────────
st.title("🏥 AI-Powered Medical Triage Assistant")
st.caption("FDP Demo — 5-Day AI Project │ ML + DL + NLP + LLM │ HuggingFace + Colab")
st.divider()

# ── Sidebar: Patient Input ────────────────────────────────────────
with st.sidebar:
    st.header("🧑‍⚕️ Patient Details")
    name    = st.text_input("Full Name", "Ramesh Kumar")
    age     = st.slider("Age", 18, 90, 55)
    sex     = st.radio("Sex", ["Male", "Female"])
    sex_int = 1 if sex == "Male" else 0
    st.divider()
    st.subheader("Vitals")
    bp      = st.slider("Blood Pressure (mmHg)", 80, 200, 160)
    chol    = st.slider("Cholesterol (mg/dL)", 100, 400, 280)
    sugar   = st.slider("Blood Sugar (mg/dL)", 70, 300, 140)
    hr      = st.slider("Heart Rate (bpm)", 50, 200, 95)
    st.divider()
    st.subheader("Complaint & Imaging")
    complaint  = st.text_area(
        "Describe symptoms",
        "Severe chest pain and breathlessness since 2 hours with sweating"
    )
    xray_file  = st.file_uploader(
        "Upload Chest X-ray (optional)", type=["jpg", "jpeg", "png"]
    )
    run_btn    = st.button("🚀 Run AI Triage", type="primary", use_container_width=True)

# ── Main Panel ────────────────────────────────────────────────────
if not run_btn:
    st.info("👈 Fill in patient details in the sidebar and click **Run AI Triage**")
    st.image(
        "https://img.icons8.com/color/200/hospital.png",
        width=150
    )

if run_btn:
    img_bytes = xray_file.read() if xray_file else None

    with st.spinner("🔄 Running all AI modules — ML → DL → NLP → LLM..."):
        agent  = TriageAgent()
        result = agent.run(
            name, age, bp, chol, sugar, hr, sex_int,
            complaint, img_bytes
        )

    st.success("✅ Triage complete!")
    st.divider()

    # ── Metric Cards ─────────────────────────────────────────────
    st.subheader("📊 AI Triage Results")
    col1, col2, col3 = st.columns(3)

    risk_color = {"HIGH": "🔴", "MODERATE": "🟡", "LOW": "🟢"}.get(
        result["risk"]["label"], "⚪"
    )
    urg_color  = {"CRITICAL": "🔴", "MODERATE": "🟡", "MILD": "🟢"}.get(
        result["nlp"]["urgency"], "⚪"
    )
    xray_color = "🔴" if result["xray"]["label"] == "PNEUMONIA" else "🟢"

    col1.metric(
        label="❤️ Cardiovascular Risk",
        value=f"{risk_color} {result['risk']['label']}",
        delta=f"{result['risk']['score']:.0%} probability"
    )
    col2.metric(
        label="🫁 X-Ray Finding",
        value=f"{xray_color} {result['xray']['label']}",
        delta=f"{result['xray']['confidence']:.0%} confidence"
    )
    col3.metric(
        label="⚡ Urgency Level",
        value=f"{urg_color} {result['nlp']['urgency']}",
    )

    st.divider()

    # ── Symptoms ──────────────────────────────────────────────────
    col_a, col_b = st.columns(2)
    with col_a:
        st.subheader("🩺 Extracted Symptoms")
        if result["nlp"]["symptoms"]:
            for s in result["nlp"]["symptoms"]:
                st.write(f"• {s.title()}")
        else:
            st.write("No specific symptoms extracted")

    with col_b:
        if xray_file:
            xray_file.seek(0)
            st.subheader("🔬 Uploaded X-Ray")
            st.image(xray_file, width=250)

    st.divider()

    # ── AI Clinical Report ────────────────────────────────────────
    st.subheader("📋 AI Clinical Summary")
    st.info(result["report"])

    # ── Patient Summary Card ──────────────────────────────────────
    st.divider()
    with st.expander("📄 Full Patient Summary"):
        st.write(f"**Name:** {result['name']}")
        st.write(f"**Age:** {result['age']} years")
        st.write(f"**BP:** {bp} mmHg  |  **Cholesterol:** {chol} mg/dL")
        st.write(f"**Blood Sugar:** {sugar} mg/dL  |  **Heart Rate:** {hr} bpm")
        st.write(f"**Complaint:** {complaint}")
        st.write(f"**Risk Score:** {result['risk']['score']:.2f}")
        st.write(f"**X-Ray Confidence:** {result['xray']['confidence']:.2f}")
'''

with open(f'{WORK_DIR}/app.py', 'w') as f:
    f.write(app_code)
print('✅ app.py written')

✅ app.py written


In [ ]:
# ── Cell 7: Write requirements.txt ───────────────────────────────
req = """streamlit
torch
transformers
tensorflow
scikit-learn
joblib
pillow
spacy
numpy
accelerate
"""

with open(f'{WORK_DIR}/requirements.txt', 'w') as f:
    f.write(req)
print('✅ requirements.txt written')

✅ requirements.txt written


In [ ]:
# ── Cell 9: Day 5 Final Summary ───────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║      DAY 5 — AGENT + DEPLOYMENT | Piyush Pandey             ║
╠══════════════════════════════════════════════════════════════╣
║  Files Created:                                              ║
║    📄 agent.py          ← TriageAgent orchestrator          ║
║    📄 app.py            ← Streamlit web UI                  ║
║    📄 requirements.txt  ← deployment dependencies           ║
║                                                              ║
║  Models Used:                                                ║
║    🤖 ML  : Random Forest      → Risk label + score         ║
║    🤖 DL  : MobileNetV2        → X-ray classification       ║
║    🤖 NLP : DistilBERT         → Urgency + symptoms         ║
║    🤖 LLM : Flan-T5-large      → Clinical report            ║
║                                                              ║
║  Pipeline:                                                   ║
║    Patient Input                                             ║
║      → predict_risk()    → HIGH / MODERATE / LOW            ║
║      → classify_xray()   → PNEUMONIA / NORMAL               ║
║      → analyze_complaint() → CRITICAL / MODERATE / MILD     ║
║      → generate_report() → 5-sentence clinical summary      ║
║      → Streamlit Dashboard (live public URL)                 ║
║                                                              ║
║  🎉 PROJECT COMPLETE — ALL 5 DAYS DONE!                     ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║      DAY 5 — AGENT + DEPLOYMENT | Piyush Pandey             ║
╠══════════════════════════════════════════════════════════════╣
║  Files Created:                                              ║
║    📄 agent.py          ← TriageAgent orchestrator          ║
║    📄 app.py            ← Streamlit web UI                  ║
║    📄 requirements.txt  ← deployment dependencies           ║
║                                                              ║
║  Models Used:                                                ║
║    🤖 ML  : Random Forest      → Risk label + score         ║
║    🤖 DL  : MobileNetV2        → X-ray classification       ║
║    🤖 NLP : DistilBERT         → Urgency + symptoms         ║
║    🤖 LLM : Flan-T5-large      → Clinical report            ║
║                                                              ║
║  Pipeline:                                                   ║
║    Patient Input                     

In [ ]:
# ── Save Day-5 files to Google Drive ─────────────────────────────

from google.colab import drive
import shutil
import os

# 1. Mount Drive (if not already mounted)
drive.mount('/content/drive')

# 2. Define paths
SOURCE_DIR = '/content/fdp_app'   # where your files are created
DEST_DIR   = '/content/drive/MyDrive/fdp_day5'  # destination folder

# 3. Create destination folder if not exists
os.makedirs(DEST_DIR, exist_ok=True)

# 4. Files to copy
files_to_copy = ['agent.py', 'app.py', 'requirements.txt']

print("\n📂 Saving files to Drive...\n")

for file in files_to_copy:
    src_path = os.path.join(SOURCE_DIR, file)
    dest_path = os.path.join(DEST_DIR, file)

    if os.path.exists(src_path):
        shutil.copy(src_path, dest_path)
        print(f"✅ Saved: {file}")
    else:
        print(f"❌ Not found: {file}")

print(f"\n🎉 All files saved in: {DEST_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📂 Saving files to Drive...

✅ Saved: agent.py
✅ Saved: app.py
✅ Saved: requirements.txt

🎉 All files saved in: /content/drive/MyDrive/fdp_day5


In [ ]:
# ── Cell: Install Gradio ──────────────────────────────────────────
!pip install -q gradio

In [ ]:
# ── Cell: Full Gradio App ─────────────────────────────────────────
import gradio as gr
import joblib, torch, numpy as np, io
import tensorflow as tf
from PIL import Image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModelForSeq2SeqLM)

BASE = '/content/drive/MyDrive/fdp'

print('Loading all models... please wait...')

# ── ML ────────────────────────────────────────────────────────────
ml_model  = joblib.load(f'{BASE}/ml_model/ml_model.pkl')
ml_scaler = joblib.load(f'{BASE}/ml_model/ml_scaler.pkl')
print('✅ ML model loaded')

# ── DL ────────────────────────────────────────────────────────────
dl_model  = tf.keras.models.load_model(f'{BASE}/dl_model/dl_model.keras')
DL_CLASSES = {0: 'NORMAL', 1: 'PNEUMONIA'}
print('✅ DL model loaded')

# ── NLP ───────────────────────────────────────────────────────────
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nlp_tokenizer = AutoTokenizer.from_pretrained(f'{BASE}/nlp_model')
nlp_model     = AutoModelForSequenceClassification.from_pretrained(
                    f'{BASE}/nlp_model').to(device)
nlp_model.eval()
NLP_LABELS    = {0: 'CRITICAL', 1: 'MODERATE', 2: 'MILD'}
SYMPTOM_WORDS = {
    'chest pain', 'breathlessness', 'fever', 'vomiting', 'headache',
    'dizziness', 'nausea', 'weakness', 'cough', 'rash', 'swelling',
    'paralysis', 'seizure', 'unconscious', 'sweating', 'bleeding',
    'burning', 'fatigue', 'palpitations', 'wheezing', 'confusion'
}
print('✅ NLP model loaded')

# ── LLM ───────────────────────────────────────────────────────────
llm_tok = AutoTokenizer.from_pretrained(f'{BASE}/llm_model')
llm_mdl = AutoModelForSeq2SeqLM.from_pretrained(f'{BASE}/llm_model').to(device)
llm_mdl.eval()
print('✅ LLM model loaded')

print(f'\n🎉 All models ready on {device}!')

Loading all models... please wait...
✅ ML model loaded


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 40 variables whereas the saved optimizer has 78 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✅ DL model loaded


Loading weights:   0%|          | 0/104 [00:02<?, ?it/s]

✅ NLP model loaded


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ LLM model loaded

🎉 All models ready on cuda!


In [ ]:
# ── Cell: Install additional dependencies ─────────────────────────
!pip install -q gradio secure-smtplib

In [ ]:
# ── Cell: Doctor Email Directory + Email Function ─────────────────
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from datetime import datetime

# ── Doctor Directory (add real emails here) ───────────────────────
DOCTOR_DIRECTORY = {
    'CARDIOLOGIST': {
        'name'  : 'Dr. Dwivedi',
        'email' : 'dwivedemergency@gmail.com',
        'dept'  : 'Cardiology Department',
        'phone' : '+91-9800000100'
    },
    'PULMONOLOGIST': {
        'name'  : 'Dr. Priya Singh',
        'email' : 'pulmonologist@hospital.com',
        'dept'  : 'Pulmonology Department',
        'phone' : '+91-9876543211'
    },
    'NEUROLOGIST': {
        'name'  : 'Dr. Piyush Pandey',
        'email' : 'emergency@gmail.com',
        'dept'  : 'Neurology Department',
        'phone' : '+91-9870000012'
    },
    'GENERAL_PHYSICIAN': {
        'name'  : 'Dr. Arjun Panday',
        'email' : 'arjunemergency@gmail.com',
        'dept'  : 'General Medicine',
        'phone' : '+91-9876543213'
    },
    'EMERGENCY': {
        'name'  : 'Dr. Satvik',
        'email' : 'satvikemergency@gmail.com',
        'dept'  : 'Emergency Department',
        'phone' : '+91-9876543214'
    }
}

# ── Gmail credentials (use App Password, not your real password) ──
SENDER_EMAIL    = 'your-gmail-address'       # ← your Gmail
SENDER_PASSWORD = 'your-gmail-app-password'        # ← Gmail App Password
                                               # Get from: Google Account
                                               # → Security → App Passwords

# ── Auto-assign doctor based on findings ─────────────────────────
def assign_doctor(risk, xray, nlp):
    urgency = nlp.get('urgency', 'MILD')
    xray_label = xray.get('label', 'NORMAL')
    risk_label = risk.get('label', 'LOW')
    symptoms   = nlp.get('symptoms', [])

    # CRITICAL always goes to emergency first
    if urgency == 'CRITICAL':
        if xray_label == 'PNEUMONIA':
            return 'PULMONOLOGIST'
        if risk_label == 'HIGH':
            return 'CARDIOLOGIST'
        return 'EMERGENCY'

    # MODERATE — route by finding
    if urgency == 'MODERATE':
        if xray_label == 'PNEUMONIA':
            return 'PULMONOLOGIST'
        if risk_label in ['HIGH', 'MODERATE']:
            return 'CARDIOLOGIST'
        if any(s in symptoms for s in ['headache', 'paralysis', 'seizure', 'confusion']):
            return 'NEUROLOGIST'
        return 'GENERAL_PHYSICIAN'

    # MILD — general physician
    return 'GENERAL_PHYSICIAN'

In [ ]:
# ── Cell: Email HTML Template ─────────────────────────────────────
def build_email_html(patient_name, age, sex, bp, chol, sugar, hr,
                     complaint, risk, xray, nlp, report,
                     doctor_info, specialist_type):

    urgency_color = {
        'CRITICAL': '#FF0000',
        'MODERATE': '#FF8C00',
        'MILD':     '#28A745'
    }.get(nlp['urgency'], '#6C757D')

    risk_color = {
        'HIGH':     '#FF0000',
        'MODERATE': '#FF8C00',
        'LOW':      '#28A745'
    }.get(risk['label'], '#6C757D')

    syms_html = ''.join(
        f'<li style="margin:4px 0;">{s.title()}</li>'
        for s in nlp['symptoms']
    ) or '<li>None detected</li>'

    timestamp = datetime.now().strftime('%d %B %Y, %I:%M %p')

    return f"""
    <html>
    <body style="font-family: Arial, sans-serif; max-width: 700px;
                 margin: auto; background: #f4f4f4; padding: 20px;">

      <!-- Header -->
      <div style="background: #1A237E; color: white; padding: 25px;
                  border-radius: 10px 10px 0 0; text-align: center;">
        <h1 style="margin:0; font-size: 22px;">🏥 AI Medical Triage System</h1>
        <p style="margin:5px 0 0; font-size: 13px; opacity: 0.85;">
          Emergency Department — Patient Referral Alert
        </p>
        <p style="margin:5px 0 0; font-size: 12px; opacity: 0.7;">
          {timestamp}
        </p>
      </div>

      <!-- Urgency Banner -->
      <div style="background: {urgency_color}; color: white; padding: 15px;
                  text-align: center; font-size: 20px; font-weight: bold;">
        ⚡ URGENCY: {nlp['urgency']} — Immediate Attention Required
      </div>

      <!-- Doctor Info -->
      <div style="background: #E3F2FD; padding: 20px; border-left: 5px solid #1565C0;">
        <h2 style="margin:0 0 10px; color: #1565C0;">
          📋 Referral To: {doctor_info['name']}
        </h2>
        <p style="margin:3px 0;"><b>Department:</b> {doctor_info['dept']}</p>
        <p style="margin:3px 0;"><b>Specialty:</b>  {specialist_type.replace('_',' ').title()}</p>
        <p style="margin:3px 0;"><b>Contact:</b>    {doctor_info['phone']}</p>
      </div>

      <!-- Patient Details -->
      <div style="background: white; padding: 20px; margin-top: 2px;">
        <h3 style="color: #1A237E; border-bottom: 2px solid #1A237E; padding-bottom: 8px;">
          👤 Patient Information
        </h3>
        <table style="width:100%; border-collapse: collapse;">
          <tr style="background:#F5F5F5;">
            <td style="padding:8px; border:1px solid #ddd;"><b>Name</b></td>
            <td style="padding:8px; border:1px solid #ddd;">{patient_name}</td>
            <td style="padding:8px; border:1px solid #ddd;"><b>Age / Sex</b></td>
            <td style="padding:8px; border:1px solid #ddd;">{age} yrs / {sex}</td>
          </tr>
          <tr>
            <td style="padding:8px; border:1px solid #ddd;"><b>Blood Pressure</b></td>
            <td style="padding:8px; border:1px solid #ddd;">{bp} mmHg</td>
            <td style="padding:8px; border:1px solid #ddd;"><b>Heart Rate</b></td>
            <td style="padding:8px; border:1px solid #ddd;">{hr} bpm</td>
          </tr>
          <tr style="background:#F5F5F5;">
            <td style="padding:8px; border:1px solid #ddd;"><b>Cholesterol</b></td>
            <td style="padding:8px; border:1px solid #ddd;">{chol} mg/dL</td>
            <td style="padding:8px; border:1px solid #ddd;"><b>Blood Sugar</b></td>
            <td style="padding:8px; border:1px solid #ddd;">{sugar} mg/dL</td>
          </tr>
        </table>

        <div style="background:#FFF9C4; padding:12px; margin-top:15px;
                    border-radius:5px; border-left: 4px solid #F9A825;">
          <b>Patient Complaint:</b><br>{complaint}
        </div>
      </div>

      <!-- AI Results -->
      <div style="background: white; padding: 20px; margin-top: 2px;">
        <h3 style="color: #1A237E; border-bottom: 2px solid #1A237E; padding-bottom: 8px;">
          🤖 AI Diagnostic Results
        </h3>
        <table style="width:100%; border-collapse: collapse;">
          <tr>
            <td style="padding:12px; border:1px solid #ddd; width:33%;">
              <div style="text-align:center;">
                <div style="font-size:13px; color:#666;">❤️ Cardiovascular Risk</div>
                <div style="font-size:22px; font-weight:bold; color:{risk_color};">
                  {risk['label']}
                </div>
                <div style="font-size:12px; color:#888;">
                  Score: {risk['score']:.0%}
                </div>
              </div>
            </td>
            <td style="padding:12px; border:1px solid #ddd; width:33%;">
              <div style="text-align:center;">
                <div style="font-size:13px; color:#666;">🫁 X-Ray Finding</div>
                <div style="font-size:22px; font-weight:bold;
                     color:{'#FF0000' if xray['label']=='PNEUMONIA' else '#28A745'};">
                  {xray['label']}
                </div>
                <div style="font-size:12px; color:#888;">
                  Confidence: {xray['confidence']:.0%}
                </div>
              </div>
            </td>
            <td style="padding:12px; border:1px solid #ddd; width:33%;">
              <div style="text-align:center;">
                <div style="font-size:13px; color:#666;">⚡ Urgency Level</div>
                <div style="font-size:22px; font-weight:bold; color:{urgency_color};">
                  {nlp['urgency']}
                </div>
                <div style="font-size:12px; color:#888;">NLP Classification</div>
              </div>
            </td>
          </tr>
        </table>

        <!-- Symptoms -->
        <div style="margin-top:15px;">
          <b>🩺 Extracted Symptoms:</b>
          <ul style="margin:8px 0; padding-left:20px; color:#333;">
            {syms_html}
          </ul>
        </div>
      </div>

      <!-- AI Clinical Report -->
      <div style="background: #E8F5E9; padding: 20px; margin-top: 2px;
                  border-left: 5px solid #2E7D32;">
        <h3 style="color: #2E7D32; margin:0 0 12px;">📋 AI Clinical Summary</h3>
        <p style="line-height: 1.8; color: #333; margin:0;">{report}</p>
      </div>

      <!-- Footer -->
      <div style="background: #1A237E; color: white; padding: 15px;
                  border-radius: 0 0 10px 10px; text-align: center;
                  font-size: 12px; margin-top: 2px;">
        <p style="margin:0;">
          ⚠️ This is an AI-generated triage alert. Clinical judgment must be applied.
        </p>
        <p style="margin:5px 0 0; opacity:0.7;">
          AI Medical Triage System — FDP Demo | Powered by ML + DL + NLP + LLM
        </p>
      </div>

    </body>
    </html>
    """

In [ ]:
# ── Cell: Send Email Function ─────────────────────────────────────
def send_referral_email(patient_name, age, sex, bp, chol, sugar, hr,
                        complaint, risk, xray, nlp, report):
    try:
        # Assign doctor
        specialist_type = assign_doctor(risk, xray, nlp)
        doctor_info     = DOCTOR_DIRECTORY[specialist_type]

        # Build email
        msg = MIMEMultipart('alternative')
        msg['From']    = f'Emergency Department <{SENDER_EMAIL}>'
        msg['To']      = doctor_info['email']
        msg['Subject'] = (
            f"🚨 [{nlp['urgency']}] Triage Alert — {patient_name}, {age}F/{sex[0]} "
            f"→ {specialist_type.replace('_',' ').title()} Referral"
        )

        # Also CC emergency if CRITICAL
        if nlp['urgency'] == 'CRITICAL':
            msg['Cc'] = DOCTOR_DIRECTORY['EMERGENCY']['email']

        html_body = build_email_html(
            patient_name, age, sex, bp, chol, sugar, hr,
            complaint, risk, xray, nlp, report,
            doctor_info, specialist_type
        )
        msg.attach(MIMEText(html_body, 'html'))

        # Send via Gmail SMTP
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(SENDER_EMAIL, SENDER_PASSWORD)
            recipients = [doctor_info['email']]
            if nlp['urgency'] == 'CRITICAL':
                recipients.append(DOCTOR_DIRECTORY['EMERGENCY']['email'])
            server.sendmail(SENDER_EMAIL, recipients, msg.as_string())

        return (
            f"✅ Email sent successfully!\n\n"
            f"📨 Referred To  : {doctor_info['name']}\n"
            f"🏥 Department   : {doctor_info['dept']}\n"
            f"📧 Email        : {doctor_info['email']}\n"
            f"⚡ Urgency      : {nlp['urgency']}\n"
            f"🕐 Sent At      : {datetime.now().strftime('%d %b %Y, %I:%M %p')}\n"
            + (f"\n📋 CC'd Emergency Department (CRITICAL case)"
               if nlp['urgency'] == 'CRITICAL' else "")
        )

    except smtplib.SMTPAuthenticationError:
        return (
            "❌ Gmail Authentication Failed!\n\n"
            "Please check:\n"
            "1. SENDER_EMAIL is correct\n"
            "2. SENDER_PASSWORD is an App Password (not your Gmail password)\n"
            "   → Google Account → Security → 2-Step Verification → App Passwords"
        )
    except Exception as e:
        return f"❌ Email failed: {str(e)}"

In [ ]:
# ── Cell: Full Gradio App with Email Button ───────────────────────
import gradio as gr

# Store triage result globally to pass to email function
triage_state = {}

def run_triage(name, age, sex, bp, chol, sugar, hr, complaint, xray_img):
    global triage_state
    sex_int = 1 if sex == 'Male' else 0

    # ── ML Risk ───────────────────────────────────────────────────
    try:
        feat  = [[age, sex_int, 0, bp, chol, sugar, 0, hr, 0, 1.0, 1, 0, 2]]
        feat  = ml_scaler.transform(feat)
        score = ml_model.predict_proba(feat)[0][1]
        label = 'HIGH' if score > 0.6 else 'MODERATE' if score > 0.35 else 'LOW'
        risk  = {'label': label, 'score': round(float(score), 2)}
    except:
        risk  = {'label': 'UNKNOWN', 'score': 0.0}

    # ── DL X-ray ──────────────────────────────────────────────────
    if xray_img is not None:
        try:
            img  = Image.fromarray(xray_img).convert('RGB').resize((224, 224))
            arr  = preprocess_input(np.array(img, dtype=np.float32))
            arr  = np.expand_dims(arr, 0)
            pred = dl_model.predict(arr, verbose=0)
            idx  = int(np.argmax(pred))
            xray = {'label': DL_CLASSES[idx],
                    'confidence': round(float(np.max(pred)), 2)}
        except:
            xray = {'label': 'Error', 'confidence': 0.0}
    else:
        xray = {'label': 'Not provided', 'confidence': 0.0}

    # ── NLP ───────────────────────────────────────────────────────
    try:
        enc = nlp_tokenizer([complaint], padding=True, truncation=True,
                            max_length=128, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = nlp_model(**enc).logits
        pid  = int(torch.argmax(logits))
        syms = list({w for w in SYMPTOM_WORDS if w in complaint.lower()})
        nlp  = {'symptoms': syms, 'urgency': NLP_LABELS[pid]}
    except:
        nlp  = {'symptoms': [], 'urgency': 'UNKNOWN'}

    # ── LLM ───────────────────────────────────────────────────────
    try:
        syms_str = ', '.join(nlp['symptoms']) if nlp['symptoms'] else 'none reported'
        urgency_desc = {
            'CRITICAL': 'life-threatening, requires immediate intervention',
            'MODERATE': 'serious but stable, requires prompt attention',
            'MILD':     'non-urgent, can be managed with routine care'
        }.get(nlp['urgency'], nlp['urgency'])
        prompt = (
            f'You are a senior emergency medicine physician. '
            f'Patient {name}, age {age}. '
            f'Risk: {risk["label"]} ({risk["score"]:.0%}). '
            f'X-ray: {xray["label"]} ({xray["confidence"]:.0%}). '
            f'Symptoms: {syms_str}. Urgency: {nlp["urgency"]} — {urgency_desc}. '
            f'Write a 5-sentence clinical triage summary and recommend action.'
        )
        inp = llm_tok(prompt, return_tensors='pt',
                      truncation=True, max_length=600).to(device)
        with torch.no_grad():
            out = llm_mdl.generate(
                **inp, max_new_tokens=300, min_new_tokens=80,
                num_beams=5, length_penalty=2.0,
                no_repeat_ngram_size=3, repetition_penalty=1.3
            )
        report = llm_tok.decode(out[0], skip_special_tokens=True).strip()
        if len(report.split()) < 20:
            raise ValueError('Too short')
    except:
        action = {
            'CRITICAL': 'Immediate transfer to emergency resuscitation bay required.',
            'MODERATE': 'Admit for monitoring and treatment.',
            'MILD':     'Discharge with outpatient follow-up within 48 hours.'
        }.get(nlp['urgency'], 'Clinical evaluation recommended.')
        report = (
            f'Patient {name}, {age} years, presents with {risk["label"].lower()} '
            f'cardiovascular risk and {xray["label"].lower()} on chest imaging. '
            f'Symptoms: {syms_str if nlp["symptoms"] else "none"}. '
            f'Urgency: {nlp["urgency"]}. {action}'
        )

    # ── Save state for email ───────────────────────────────────────
    triage_state.update({
        'name': name, 'age': age, 'sex': sex,
        'bp': bp, 'chol': chol, 'sugar': sugar, 'hr': hr,
        'complaint': complaint, 'risk': risk,
        'xray': xray, 'nlp': nlp, 'report': report
    })

    # ── Assign doctor preview ──────────────────────────────────────
    specialist  = assign_doctor(risk, xray, nlp)
    doctor_info = DOCTOR_DIRECTORY[specialist]

    # ── Format outputs ─────────────────────────────────────────────
    risk_icon = {'HIGH':'🔴','MODERATE':'🟡','LOW':'🟢'}.get(risk['label'],'⚪')
    xray_icon = '🔴' if xray['label'] == 'PNEUMONIA' else '🟢'
    urg_icon  = {'CRITICAL':'🔴','MODERATE':'🟡','MILD':'🟢'}.get(nlp['urgency'],'⚪')

    doctor_preview = (
        f"📨 Will be sent to: {doctor_info['name']}\n"
        f"🏥 Department    : {doctor_info['dept']}\n"
        f"📧 Email         : {doctor_info['email']}\n"
        f"🔬 Specialty     : {specialist.replace('_',' ').title()}"
        + (f"\n📋 CC: Emergency Department (CRITICAL case)"
           if nlp['urgency'] == 'CRITICAL' else "")
    )

    return (
        f"{risk_icon} {risk['label']}  ({risk['score']:.0%} probability)",
        f"{xray_icon} {xray['label']}  ({xray['confidence']:.0%} confidence)",
        f"{urg_icon} {nlp['urgency']}",
        '\n'.join([f'• {s.title()}' for s in nlp['symptoms']]) or 'None detected',
        report,
        doctor_preview,
        gr.update(visible=True)   # show email button
    )

def send_email_action():
    if not triage_state:
        return '❌ Run triage first before sending email'
    return send_referral_email(
        triage_state['name'],    triage_state['age'],
        triage_state['sex'],     triage_state['bp'],
        triage_state['chol'],    triage_state['sugar'],
        triage_state['hr'],      triage_state['complaint'],
        triage_state['risk'],    triage_state['xray'],
        triage_state['nlp'],     triage_state['report']
    )

# ── Build UI ──────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title='AI Medical Triage') as demo:

    gr.Markdown("""
    # 🏥 AI-Powered Medical Triage Assistant
    **FDP Demo — Day 5 | ML + DL + NLP + LLM + Email Agent**
    """)

    with gr.Row():
        # ── Inputs ───────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown('### 🧑‍⚕️ Patient Details')
            name      = gr.Textbox(label='Full Name',  value='Ramesh Kumar')
            age       = gr.Slider(18, 90, value=55,    label='Age')
            sex       = gr.Radio(['Male','Female'],     label='Sex', value='Male')

            gr.Markdown('### 💉 Vitals')
            bp        = gr.Slider(80,  200, value=160, label='Blood Pressure (mmHg)')
            chol      = gr.Slider(100, 400, value=280, label='Cholesterol (mg/dL)')
            sugar     = gr.Slider(70,  300, value=140, label='Blood Sugar (mg/dL)')
            hr        = gr.Slider(50,  200, value=95,  label='Heart Rate (bpm)')

            gr.Markdown('### 📝 Complaint & Imaging')
            complaint = gr.Textbox(
                label='Describe Symptoms',
                value='Severe chest pain and breathlessness since 2 hours',
                lines=3
            )
            xray_img  = gr.Image(label='Upload Chest X-Ray (optional)', type='numpy')
            run_btn   = gr.Button('🚀 Run AI Triage', variant='primary', size='lg')

        # ── Outputs ──────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown('### 📊 AI Triage Results')

            with gr.Row():
                risk_out = gr.Textbox(label='❤️ Cardiovascular Risk', interactive=False)
                xray_out = gr.Textbox(label='🫁 X-Ray Finding',        interactive=False)

            with gr.Row():
                urg_out  = gr.Textbox(label='⚡ Urgency Level',         interactive=False)
                syms_out = gr.Textbox(label='🩺 Symptoms', lines=4,    interactive=False)

            gr.Markdown('### 📋 AI Clinical Summary')
            report_out = gr.Textbox(label='Clinical Report', lines=7,  interactive=False)

            gr.Markdown('### 📨 Doctor Referral')
            doctor_out = gr.Textbox(
                label='Assigned Doctor Preview',
                lines=5, interactive=False
            )

            # Email button — hidden until triage runs
            email_btn = gr.Button(
                '📧 Send Referral Email to Doctor',
                variant='primary',
                visible=False,
                size='lg'
            )
            email_status = gr.Textbox(
                label='📬 Email Status',
                interactive=False,
                lines=6
            )

    # ── Examples ─────────────────────────────────────────────────
    gr.Markdown('### 🧪 Quick Test Examples')
    gr.Examples(
        examples=[
            ['Ramesh Kumar', 55, 'Male',   160, 280, 140, 95,
             'Severe chest pain and breathlessness since 2 hours', None],
            ['Sunita Devi',  42, 'Female', 130, 210, 110, 80,
             'High fever with vomiting and severe dizziness', None],
            ['Arjun Singh',  28, 'Male',   115, 180,  90, 72,
             'Mild cold and cough for 2 days, no fever', None],
        ],
        inputs=[name, age, sex, bp, chol, sugar, hr, complaint, xray_img]
    )

    # ── Wire buttons ──────────────────────────────────────────────
    run_btn.click(
        fn=run_triage,
        inputs=[name, age, sex, bp, chol, sugar, hr, complaint, xray_img],
        outputs=[risk_out, xray_out, urg_out, syms_out,
                 report_out, doctor_out, email_btn]
    )

    email_btn.click(
        fn=send_email_action,
        inputs=[],
        outputs=[email_status]
    )

demo.launch(share=True, debug=False)

/tmp/ipykernel_535/1084419149.py:138: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title='AI Medical Triage') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://30400f2e4c83983345.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
